# Tutorial 4: Deep Learning Basics with PyTorch

<a id="toc"></a>

This tutorial introduces the **end-to-end workflow of training neural networks in PyTorch**:
tensors -> automatic differentiation -> model definition -> training loop -> evaluation.

You’ll start with small, concrete examples (so you can *see* what gradients are doing), then train an **Multilayer Perceptron** (MLP) on MNIST, and finally apply the same ideas to a **regression** task.

At the end, there is an **Optional Advanced** section with extra experiments (deeper gradient demonstrations and Convolutional Neural Networks (CNNs)). It’s there for curiosity and extension, note that the main tutorial does **not** depend on it.

\
**Learning goals**
- Understand tensor shapes and why they matter (most bugs are shape bugs!)
- Use **autograd** to compute gradients and reason about backpropagation
- Write a clean training loop: forward -> loss -> backward -> optimizer step -> evaluate
- Train an MLP classifier and interpret loss/accuracy
- Reuse the same training pattern for regression

\
**How to use this notebook**
- Run cells **top-to-bottom**.
- If you have a GPU in Colab: *Runtime -> Change runtime type -> GPU* (optional).
- When you see **Task** prompts, try them first, then check the provided solution.

<br>

---

## Table of Contents
1. [Setup](#1-setup)
2. [Tensors and Autograd: Computing Gradients](#2-tensors-and-autograd-computing-gradients)
3. [Multilayer Perceptron (MLP) for MNIST](#3-multilayer-perceptron-mlp-for-mnist)
4. [Exercise: Regression with an MLP](#4-exercise-regression-with-an-mlp)
5. [Optional Advanced: Gradients + CNNs](#5-optional-advanced-gradients--cnns)


## 1. Setup
<a id="sec1"></a>

[⬆️ Back to Table of Contents](#toc)

We start by installing/importing PyTorch, choosing CPU vs GPU

- Colab setup: install PyTorch/torchvision if needed (may already be installed).
If you are using a local environment, you can skip this step if you have already installed PyTorch.
- You can also specify a particular version of PyTorch if your code requires it.
- Also, it can be a good practice to use a ***virtual environment*** for your project to manage dependencies and avoid conflicts between different projects.
  - You can create a virtual environment using tools like `venv` or `conda`, and then install the required packages within that environment. This way, you can ensure that your project has the correct versions of libraries without affecting other projects on your system.


In [1]:
!pip install torch torchvision # pip often install the latest version of torch, which may not be compatible with your code. You can specify a version if needed, e.g., !pip install torch==1.9.0 torchvision==0.10.0
                               # torchvision is a package that provides popular datasets, model architectures, and image transformations for computer vision tasks. It is often used in conjunction with PyTorch for building and training neural networks on image data


In [2]:
# import torch and print the version we are using.
import torch
print("Using torch", torch.__version__)

Using torch 2.10.0+cpu


By default, all operations are performed using [CPU](https://en.wikipedia.org/wiki/Central_processing_unit). For a much faster performance, we can instead use [GPU](https://en.wikipedia.org/wiki/Graphics_processing_unit). First, we check if CUDA is available, if so, we set the device to `cuda` (otherwise, we stick with CPU).

If you're using Colab and the below code prints "Device: CPU", you can change the runtime type to GPU (or [TPU](https://en.wikipedia.org/wiki/Tensor_Processing_Unit), if you fancy it).

If you are using Mac with M-series chips, you can use torch.backends.mps.is_available() to check if you can use the GPU. If it returns True, you can set device = torch.device("mps") to use the GPU (MPS short for Metal Performance Shaders). However, note that **not all** operations are supported on MPS, so you may encounter some issues when using it. You can refer to the PyTorch and Mac documentation for more details on MPS support: https://pytorch.org/docs/stable/notes/mps.html; https://developer.apple.com/metal/pytorch/

In [3]:
# Check whether a GPU is available and choose the device accordingly
# Using a GPU can significantly speed up training

gpu_avail = torch.cuda.is_available()
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Device: {device}")

Device: cpu


## 2. Tensors and Autograd: Computing Gradients
<a id="sec2"></a>

[⬆️ Back to Table of Contents](#toc)

In this part, we refresh the definitions of tensors and their ranks (scalar/vector/matrix), then explore how PyTorch tracks operations and computes gradients using **autograd**.

<br>

### 2.1 Tensors

A tensor is an algebraic object that generalizes vectors and matrices to higher dimensions. Tensors have ranks (a.k.a. number of axes):
- Rank 0: Scalar (a single number); Shape: `()`
- Rank 1: Vector (one-dimensional array); Shape: `(n,)`
- Rank 2: Matrix (two-dimensional array); Shape: `(m,n)`
- Higher-order tensors 3+, i.e., Rank $k$: Multidimensional arrays; Shape: `(d1, d2, ..., dk)`

\
Important:
- **rank** (ndim) = `len(shape)`, number of axes
- **shape** = sizes along *each* axis

\
In deep learning (DL), tensors are often used to represent data because GPUs and TPUs can process tensors efficiently.

<br>

> **Intuition: tensors are just n-dimensional arrays**  
> - The **shape** tells you how data is organised (batch, channels, height, width, …).  
> - Most DL errors come from mismatched shapes, when debugging, print shapes early and often.  
> - In PyTorch, gradients are tracked only for tensors with `requires_grad=True`.

In [4]:
# examples
a = torch.tensor(3.0)           # scalar
b = torch.tensor([1.0, 2.0])    # vector
c = torch.zeros(2, 3)           # matrix
d = torch.zeros(4, 2, 3)        # rank-3 tensor

print("a:", a.shape, "rank:", a.ndim)
print("b:", b.shape, "rank:", b.ndim)
print("c:", c.shape, "rank:", c.ndim)
print("d:", d.shape, "rank:", d.ndim)

a: torch.Size([]) rank: 0
b: torch.Size([2]) rank: 1
c: torch.Size([2, 3]) rank: 2
d: torch.Size([4, 2, 3]) rank: 3


Typical ML/DL tensor shapes
- Tabular batch: `(batch, features)`
- Sequence batch: `(batch, time, features)` or `(time, batch, features)`
- Images (PyTorch): `(batch, channels, height, width)`
- Videos: `(batch, channels, time, height, width)`

Each axis is just an indexing direction, after 3D we stop visualizing physically and think them as "multi-index arrays".


In [5]:
# Also, when new to tensor, you may easily get confused with scalar, a vector with one component, and matrix with one element, for example:

scalar = torch.tensor(3.0)      # rank 0
vector1 = torch.tensor([3.0])   # rank 1
matrix1 = torch.tensor([[3.0]]) # rank 2

print("scalar:", scalar.shape, scalar.ndim)
print("vector1:", vector1.shape, vector1.ndim)
print("matrix1:", matrix1.shape, matrix1.ndim)

scalar: torch.Size([]) 0
vector1: torch.Size([1]) 1
matrix1: torch.Size([1, 1]) 2


In the example above, even though all of them only contains one number 3.0, they represent tensors with different ranks...

In DL, a lot of bugs come from tensor shape mismatches, you might want to make sure you understand these concepts clearly before playing around with more involved examples...

For the example above

| Tensor | Shape   | Rank |
| ------ | ------- | ---- |
| scalar | `()`    | 0    |
| vector | `(1,)`  | 1    |
| matrix | `(1,1)` | 2    |

**Rank ≠ number of elements.** Rank = number of *indexing* directions.

In [6]:
# one more example of matrix and scalar
matrix = torch.rand(1,1)
print(matrix[0][0])   # two indices -> rank 2

scalar = torch.tensor(5.0)
print(scalar.item())  # no indices -> rank 0
                      # .item() is used to get the Python scalar value from a single-element tensor.
                      # if you want to see the value as a tensor instead of a scalar value, you can simply print without calling .item()
print(scalar)
print(matrix)         # in the printed result, you can clearly tell the rank as well by checking the [] around it


tensor(0.0717)
5.0
tensor(5.)
tensor([[0.0717]])


### 2.2 Autograd intuition

Autograd builds a **computation graph** as you do operations on tensors with `requires_grad=True`.

- Each operation creates a node in the graph
- Autograd stores enough information to apply the **chain rule** backward
- When you call `.backward()`, PyTorch walks the graph in reverse and accumulates gradients into `.grad`

*Key idea*: autograd is efficient because it computes gradients **without explicitly building full Jacobians** for large systems.

In [7]:
# Let's start by checking a tiny graph

# by default, requires_grad is False, which means that PyTorch will not track operations on this tensor for automatic differentiation
x1 = torch.tensor(2.0)  # creat a rank 0 tensor (scalar) with value 2.0
print("x1 Gradient tracking:", x1.requires_grad)

# If you want to enable gradient tracking for a tensor, you can set requires_grad=True when creating the tensor, e.g., x = torch.rand(2.0, requires_grad=True)
x2 = torch.tensor(2.0, requires_grad=True)
print("x2 Gradient tracking:", x2.requires_grad)

y = x2**2 + 3*x2 + 1 # x2 is used in the computation of y, so PyTorch will track the operations on x2 and compute gradients with respect to it during backpropagation
print("y =", y.item()) # .item() is used to get the Python scalar value from a single-element tensor. This is useful for printing or using the value in other computations without the overhead of a tensor
print("y =", y) # if you want to see the value of y as a tensor instead of a scalar value, you can simply print y without calling .item().
                # This will show the value of y as a tensor, which can be useful for debugging or when working with higher-dimensional tensors where the value may not be a single scalar.
                # in this case, y is a rank 0 tensor (scalar) with the computed value, and printing it directly will show the tensor representation of that scalar value
print("shape of y:", y.shape, "rank:", y.ndim)
y.backward() # This will compute the gradients of y with respect to x2 and store it in x2.grad.
print("dy/dx =", x2.grad.item())
print("dy/dx =", x2.grad)

x1 Gradient tracking: False
x2 Gradient tracking: True
y = 11.0
y = tensor(11., grad_fn=<AddBackward0>)
shape of y: torch.Size([]) rank: 0
dy/dx = 7.0
dy/dx = tensor(7.)


We had:  
$y = x_2^2 + 3x_2 + 1$, with $ \frac{dy}{dx_2} = 2x_2 + 3$

\
At $x_2=2.0$, we have $2 * 2.0 + 3 = 7.0$, which maatches what PyTorch obtained.


***Also, don't confuse Python scalar with PyTorch scalar***

Python scalar:
- No shape
- No gradient tracking
- No device (CPU/GPU/...)

PyTorch 0-d tensor:
- Shape `()`
- Can require gradients
- Can on GPU (e.g., `cuda`, `mps`, `xpu`)/CPU
- Part of computation graph

***Why True 0-D Tensors Matter***

In DL, a loss function is often involved, and loss values in neural networks must:
- Participate in autograd
- Track gradient history
- Live on the device, GPU/CPU
- Support dtype/device
- Therefore, PyTorch represents scalars as rank-0 tensors, not Python floats

Make sure to declare them with the correct data type

In [8]:
a = 3.0                # Python float
b = torch.tensor(3.0, requires_grad=True)  # PyTorch 0-d tensor, i.e., rank 0

print(type(a))
print(type(b))
print(b.shape)

c = 2 * a
d = 2 * b

d.backward()

try:
    c.backward()
except:
    print("c has no attribute 'backward'")

<class 'float'>
<class 'torch.Tensor'>
torch.Size([])
c has no attribute 'backward'


### 2.3 Gradients, Jacobians, Hessians (Optional)

*This section mainly explains how to use `.backward()`, specifically, when we need to pass argument ($v$) to it. For now, you just need to **remember that `backward()` computes gradients of the scalar output w.r.t. tensors with requires_grad=True. For a vector/matrix output, you need to reduce it to a scalar first (e.g., mean or sum) if you are not passing any argument to it.***

---

\
For people interested in knowing the details, you can read through the following materials.

\
Let’s examine **gradients, Jacobians, and Hessians**, focusing on their mathematical definitions and what their **tensor shapes** look like in PyTorch.

\
*Derivatives and Shapes*

Let:

* Input: $ x \in \mathbb{R}^n $
* Output: $ y \in \mathbb{R}^m $

We distinguish between **output dimension** ($m$) (number of output components) and **tensor rank** (number of axes).



1. Scalar Output ($ m = 1 $)
    - If the function returns a **single real number**: $y \in \mathbb{R}$
    - In PyTorch this is typically represented as a **0-dimensional tensor** (shape `()`).
    - The derivative with respect to $x$ is the **gradient**:
$
\nabla_x y \in \mathbb{R}^n
$
      - Shape: `(n,)`, Rank: 1
    - Even though the output is rank 0, the gradient is rank 1 because it contains one partial derivative per input dimension

1. Vector Output ($ m > 1 $)
    - If:
$
y \in \mathbb{R}^m,
$
the derivative becomes the **Jacobian matrix**:
$
J = \frac{\partial y}{\partial x}
\in \mathbb{R}^{m \times n}
$
      - Shape: `(m, n)`, Rank: 2
    - Each output component depends on each input component, so we need two indices.

1. Second Derivatives (Hessians)
    - If:
$y \in \mathbb{R}$,
the Hessian is:
$H = \frac{\partial^2 y}{\partial x^2}\in \mathbb{R}^{n \times n}$
      - Shape: `(n, n)`, Rank: 2
    - If:
$y \in \mathbb{R}^m$, you can think of one Hessian per output component:
$
  \frac{\partial^2 y_k}{\partial x_i \partial x_j}
$
      - Shape: `(m, n, n)`, Rank: 3

#### 2.3.1 What `.backward()` actually requires

It is often said that `.backward()` “requires a scalar output.”
More precisely:

> `.backward()` requires that the output tensor has exactly **one element** (`numel() == 1`) if no gradient argument is provided.



`.backward()` computes a **vector–Jacobian product (VJP)**, not the full Jacobian. Thus, it requires choosing a direction in output space.

  - For:
  $
  y = f(x), \quad y \in \mathbb{R}^m
  $
  - Calling: `y.backward(v)`, computes: $J^T v$, where:
    - $J = \frac{\partial y}{\partial x}$
    - $v \in \mathbb{R}^m$ is the gradient argument (same shape as $y$)

<br>

1. Case 1: Single-Element Output (Implicit Gradient)
    - If the output tensor contains **exactly one element**, PyTorch automatically uses: $v = 1$
    - This includes:
      * A true scalar (shape `()`)
      * A tensor of shape `(1,)`
      * A tensor of shape `(1,1)`
      * Any tensor with `numel() == 1`
    - Because there is only one element, there is no ambiguity about the direction in output space
    - this works: `y.backward()`

1.  Case 2: Multi-Element Output
    - If the output tensor has **more than one element**, PyTorch cannot infer which linear combination of outputs you want.
    - You must provide: `y.backward(v)`, where $v$ has the **same shape as $y$**.
    - This $v$ specifies the direction in output space

<br>

Important Clarification

A tensor of shape `(1,)` is **not** a scalar (rank-0), but it *does* contain one element. For `.backward()`, what matters is:
number of elements = 1, i.e., `numel() == 1`, not the tensor rank.

<br>

Geometric Interpretation

- The Jacobian is a local linear map from **input space** to **output space**.
- Backward mode pulls a direction $v$ from output space back into input space using $J^T v$
- If output space is 1-dimensional (or has only one element), only one direction exists -> no ambiguity.
- If output space has multiple elements, you must specify which direction to propagate.

In [9]:
# Input in R^2
x = torch.tensor([2.0, 3.0], requires_grad=True)

# Scalar output (shape ())
y = x[0]**2 + 3*x[1]

print("y shape:", y.shape)   # ()
print("y numel:", y.numel())

# Backward works automatically
y.backward()

print("Gradient of y:", x.grad)

y shape: torch.Size([])
y numel: 1
Gradient of y: tensor([4., 3.])


In [10]:
from torch.autograd.functional import jacobian

def f(x):
    return torch.stack([x[0]**2, 3*x[1]])

x = torch.tensor([2.0, 3.0], requires_grad=True)

J = jacobian(f, x)

print("Jacobian:")
print(J)
print("Jacobian shape:", J.shape)


Jacobian:
tensor([[4., 0.],
        [0., 3.]])
Jacobian shape: torch.Size([2, 2])


In [11]:
x = torch.tensor([1.0, 2.0], requires_grad=True)

f = x[0]**3 + x[0]*x[1]

# First derivative
grad = torch.autograd.grad(f, x, create_graph=True)[0]

# Second derivative (Hessian)
H_rows = []
for i in range(x.numel()):
    row = torch.autograd.grad(grad[i], x, retain_graph=True)[0]
    H_rows.append(row)

H = torch.stack(H_rows)

print("Hessian:")
print(H)
print("Hessian shape:", H.shape)


Hessian:
tensor([[6., 1.],
        [1., 0.]])
Hessian shape: torch.Size([2, 2])


**Now, check the following and see if you can clearly answer these questions on your own:**

1. What tensor rank really means (and what it does NOT mean)
1. The difference between:
   - scalar `()`
   - vector `(1,)`
   - matrix `(1,1)`
1. Why `.backward()` works for scalars but not for general tensors
1. (optional) What a vector–Jacobian product is
1. (optional) How gradient, Jacobian, and Hessian relate to tensor shape

### 2.4 Examples

Let's assume we have the following function:
$$ y = x_1 + x_2 $$

To compute the gradient, we need to find the derivatvies:
$$ \frac{dy}{dx_1}, \quad \frac{dy}{dx_2} $$

Let's create two variables, *x1* and *x2* to store two tensors. We also need to find out if the gradients are being tracked as this process is crucial for training a model.

We will also perform a simple tensor addition operation.

#### 2.4.1 Case 1: Rank 0 Tensor



In [12]:
#create random tensors of Rank 0 and print their requires_grad property
x1 = torch.rand(())
x2 = torch.rand(())

# by default, requires_grad is False, which means that PyTorch will not track operations on this tensor for automatic differentiation.
print("X1 Gradient tracking:", x1.requires_grad)
print("X2 Gradient tracking:", x2.requires_grad)

y = x1 + x2 #tensor addition
print("shape of Y:", y.shape, "rank:", y.ndim)

#results
print("X1 =", x1)
print("X2 =", x2)
print("X1 + X2 = Y =", y)

try:
    y.backward()
except:
    print("No gradient is tracked")

X1 Gradient tracking: False
X2 Gradient tracking: False
shape of Y: torch.Size([]) rank: 0
X1 = tensor(0.0033)
X2 = tensor(0.4746)
X1 + X2 = Y = tensor(0.4779)
No gradient is tracked


From the above output, we can tell that no gradient is being tracked. This is the default setting. Let's change that.

In [13]:
#create random tensors and print their requires_grad property
x1 = torch.rand((), requires_grad = True) # Create a random tensor with requires_grad=True, which means that PyTorch will track all operations on this tensor and compute gradients with respect to it during backpropagation. This is essential for training neural networks, as it allows us to compute the gradients needed for optimization algorithms like SGD or Adam to update the model parameters.
x2 = torch.rand((), requires_grad = True)
print("X1 Gradient tracking:", x1.requires_grad)
print("X2 Gradient tracking:", x2.requires_grad)

y = x1 + x2 #tensor addition

#results
print("X1 =", x1)
print("X2 =", x2)
print("X1 + X2 = Y =", y)

X1 Gradient tracking: True
X2 Gradient tracking: True
X1 = tensor(0.0469, requires_grad=True)
X2 = tensor(0.5628, requires_grad=True)
X1 + X2 = Y = tensor(0.6097, grad_fn=<AddBackward0>)


The output `grad_fn` tells us that PyTorch now knows how to compute the gradient of $Y$ (it would be with respect to $x_1$ and $x_2$). We can now perform backpropagation, which is how most neural networks are trained. As part of backprop, we calculate the partial derivatives (outlined above). The function `backward()` allows us to perform backprop on $Y$. Then, we can print the gradients of $x_1$ and $x_2$.

In [14]:
# backward() computes gradients of the scalar output w.r.t. tensors with requires_grad=True
# Gradients are stored in x1.grad and x2.grad

y.backward()
print(x1.grad)
print(x2.grad)

tensor(1.)
tensor(1.)


#### 2.4.2 Case 2: Rank 1 Tensor
Let's generate a new set of tensors with Rank 1

In [15]:
#create random tensors and print their requires_grad property
x1 = torch.rand(1, requires_grad = True) # Create a random tensor of shape (1, 1) with requires_grad=True, which means that PyTorch will track all operations on this tensor and compute gradients with respect to it during backpropagation. This is essential for training neural networks, as it allows us to compute the gradients needed for optimization algorithms like SGD or Adam to update the model parameters.
x2 = torch.rand(1, requires_grad = True)
print("X1 Gradient tracking:", x1.requires_grad)
print("X2 Gradient tracking:", x2.requires_grad)

y = x1 + x2 #tensor addition
print("shape of Y:", y.shape, "rank:", y.ndim)

#results
print("X1 =", x1)
print("X2 =", x2)
print("X1 + X2 = Y =", y)

y.backward()
print(x1.grad)
print(x2.grad)

X1 Gradient tracking: True
X2 Gradient tracking: True
shape of Y: torch.Size([1]) rank: 1
X1 = tensor([0.8497], requires_grad=True)
X2 = tensor([0.5800], requires_grad=True)
X1 + X2 = Y = tensor([1.4297], grad_fn=<AddBackward0>)
tensor([1.])
tensor([1.])


#### 2.4.3 Case 3: Rank 2 Tensor
Let's generate a new set of tensors with Rank 2

In [16]:
# Rank-2 example with one element
# We create tensors with shape (1, 1) and compute y = x1 + x2

x1 = torch.rand(1, 1, requires_grad=True) # always set requires_grad=True for tensors that you want to compute gradients for.
x2 = torch.rand(1, 1, requires_grad=True)

y = x1 + x2

print("X1", x1)
print("X2", x2)
print("Y", y)
print("shape of Y:", y.shape, "rank:", y.ndim)

# in this case even through y is rank 2, but since numel =1, the following still works
y.backward()
print(x1.grad)
print(x2.grad)

X1 tensor([[0.7763]], requires_grad=True)
X2 tensor([[0.6697]], requires_grad=True)
Y tensor([[1.4460]], grad_fn=<AddBackward0>)
shape of Y: torch.Size([1, 1]) rank: 2
tensor([[1.]])
tensor([[1.]])


In [17]:
# Rank-2 example with more than one element (a small matrix / batch of vectors)
# We create tensors with shape (1, 2) and compute y = x1 + x2 element-wise

x1 = torch.rand(1, 2, requires_grad=True) # always set requires_grad=True for tensors that you want to compute gradients for.
x2 = torch.rand(1, 2, requires_grad=True)

y = x1 + x2

print("X1", x1)
print("X2", x2)
print("Y", y)
print("shape of Y:", y.shape, "rank:", y.ndim)

# what happends if we call backward() on a non-scalar output?
try:
    y.backward() # this will raise an error because y is not a scalar. PyTorch requires the output to be a scalar for backward() to compute gradients. To fix this, we can reduce y to a scalar using a reduction operation like mean() or sum() before calling backward(), as we did in the last code block
except RuntimeError as e:
    print("Error:", e)

print(x1.grad)
print(x2.grad)

X1 tensor([[0.8535, 0.6736]], requires_grad=True)
X2 tensor([[0.9986, 0.8306]], requires_grad=True)
Y tensor([[1.8521, 1.5042]], grad_fn=<AddBackward0>)
shape of Y: torch.Size([1, 2]) rank: 2
Error: grad can be implicitly created only for scalar outputs
None
None


The shape of the tensor is different this time. The method of backprop will also change slightly:
- we need to convert $Y$ to a ***scalar*** because we can compute gradients for scalars (rank 0 tensors or tensors with numel = 1).
- For simplicity, in this example, we will calculate the *mean* of $Y$ and calculate partial derivatives of $Y$ with respect to $x_1$ and $x_2$.

In [18]:
# backward() requires a scalar or tensors with only one element. For a vector/matrix output with numel > 1, reduce it to a scalar first (e.g., mean or sum)

m = y.mean() #calc means
print(m)

m.backward() #backprop
print(x1.grad)
print(x2.grad)

tensor(1.6781, grad_fn=<MeanBackward0>)
tensor([[0.5000, 0.5000]])
tensor([[0.5000, 0.5000]])
